# COPD 1-mm pTVReg MATLAB vs MR-D-BSCP P2e: landmark-TRE CDFs

Each panel compares one COPD pair's 300 native-domain snap-to-voxel landmark TREs. pTVReg is the completed MATLAB 1-mm GPU rerun; P2e is the completed fixed DirectCP cohort. No registration is run here.

The shared x-axis keeps COPD2's heavy P2e tail visible. This is a protocol-tagged comparison, not a method-only ablation.

In [ ]:
from __future__ import annotations

import csv
import os
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PTVREG_ROOT = Path(os.environ['PTVREG_COPD_MATLAB_ROOT'])
P2E_ROOT = Path(os.environ['MRDBSCP_P2E_COPD_ROOT'])
CASES = tuple(range(1, 11))
CHALLENGE_CASE = {case: 103 + case for case in CASES}
P2E_RUN_TEMPLATE = (
    'direct_cp_32_16_8_4_ptv_lcc_isotv_copd_case{case:02d}_'
    'sched64-32-16-8-4_ds16-8-4-2-1_e400-400-400-400-900_'
    'lr0p001_w0p11_seed1'
)

def read_ptvreg(case: int) -> np.ndarray:
    path = PTVREG_ROOT / f'case{case:02d}' / f'case{case:02d}_eBH_to_iBH_deformation.mat'
    if not path.is_file():
        raise FileNotFoundError(path)
    with h5py.File(path, 'r') as handle:
        required = {'Tptv_rsz', 'TRE_all_phys', 'TRE_phys'}
        missing = required - set(handle)
        if missing:
            raise ValueError(f'{path}: missing {sorted(missing)}')
        values = np.asarray(handle['TRE_all_phys']).squeeze().astype(float)
        saved_mean = float(np.asarray(handle['TRE_phys']).squeeze())
        if np.asarray(handle['Tptv_rsz']).ndim != 4:
            raise ValueError(f'{path}: Tptv_rsz is not a dense 4-D field')
    if values.size != 300 or not np.isfinite(values).all():
        raise ValueError(f'{path}: expected 300 finite landmark errors')
    if not np.isclose(values.mean(), saved_mean, rtol=0, atol=1e-8):
        raise ValueError(f'{path}: TRE_all_phys mean disagrees with TRE_phys')
    return values

def read_p2e(case: int) -> np.ndarray:
    path = P2E_ROOT / P2E_RUN_TEMPLATE.format(case=case) / 'per_landmark_tre_native_crop.csv'
    if not path.is_file():
        raise FileNotFoundError(path)
    with path.open(newline='', encoding='utf-8') as stream:
        rows = list(csv.DictReader(stream))
    if len(rows) != 300 or 'snap_tre_mm' not in rows[0]:
        raise ValueError(f'{path}: expected 300 rows with snap_tre_mm')
    values = np.asarray([float(row['snap_tre_mm']) for row in rows])
    if not np.isfinite(values).all():
        raise ValueError(f'{path}: non-finite snap TRE')
    return values

errors = {
    case: {'pTVReg MATLAB': read_ptvreg(case), 'MR-D-BSCP P2e': read_p2e(case)}
    for case in CASES
}
errors[2]['pTVReg MATLAB'].shape, errors[2]['MR-D-BSCP P2e'].shape


In [ ]:
def summarize(values: np.ndarray) -> dict[str, float]:
    return {
        'mean TRE (mm)': values.mean(),
        'median TRE (mm)': np.median(values),
        'p75 TRE (mm)': np.percentile(values, 75),
        'p95 TRE (mm)': np.percentile(values, 95),
        'max TRE (mm)': values.max(),
        'landmarks >5 mm': int((values > 5).sum()),
    }

summary_rows = [
    {'COPD case': f'COPD{case}', 'Challenge case': CHALLENGE_CASE[case], 'method': method, **summarize(values)}
    for case, method_values in errors.items()
    for method, values in method_values.items()
]
summary = pd.DataFrame(summary_rows).set_index(['COPD case', 'method']).sort_index()
summary.round(3)


In [ ]:
COLORS = {'pTVReg MATLAB': '#1f77b4', 'MR-D-BSCP P2e': '#d62728'}
Q = 0.75
xmax = float(np.ceil(max(values.max() for method_values in errors.values() for values in method_values.values()) * 1.03))

fig, axes = plt.subplots(2, 5, figsize=(18, 7.5), sharex=True, sharey=True, constrained_layout=True)
for axis, case in zip(axes.flat, CASES):
    for method, values in errors[case].items():
        sorted_values = np.sort(values)
        cdf = np.arange(1, len(sorted_values) + 1) / len(sorted_values)
        p75 = np.percentile(sorted_values, 75)
        axis.step(sorted_values, cdf, where='post', color=COLORS[method], lw=2.1, label=method)
        axis.vlines(p75, 0, Q, color=COLORS[method], ls=':', lw=1.35)
    axis.axhline(Q, color='black', ls=':', lw=1.0)
    ptv_mean = errors[case]['pTVReg MATLAB'].mean()
    p2e_mean = errors[case]['MR-D-BSCP P2e'].mean()
    axis.set_title(f'COPD{case} / challenge {CHALLENGE_CASE[case]}\nmean: pTVReg={ptv_mean:.2f}, P2e={p2e_mean:.2f} mm')
    axis.set(xlim=(0, xmax), ylim=(0, 1.02))
    axis.grid(alpha=0.22)

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=True, bbox_to_anchor=(0.5, 1.03))
fig.supxlabel('Native-domain snap-to-voxel landmark TRE (mm)')
fig.supylabel('Cumulative distribution')
fig.suptitle('COPD 1-mm: pTVReg MATLAB vs MR-D-BSCP P2e (300 landmarks per pair)', y=1.11, fontsize=15)
plt.show()


## COPD2 tail diagnostic

COPD2 / challenge case 105 has identical median TRE for both methods, so the relevant difference is in the upper tail rather than a uniform landmark shift. The full-scale and 0–6 mm views below make that distinction explicit.

In [ ]:
case = 2
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), sharey=True, constrained_layout=True)
for axis, limit, title in zip(axes, (xmax, 6), ('full CDF', '0–6 mm CDF')):
    for method, values in errors[case].items():
        sorted_values = np.sort(values)
        axis.step(sorted_values, np.arange(1, len(sorted_values) + 1) / len(sorted_values), where='post',
                  color=COLORS[method], lw=2.4, label=method)
        axis.vlines(np.percentile(sorted_values, 75), 0, Q, color=COLORS[method], ls=':', lw=1.5)
    axis.axhline(Q, color='black', ls=':', lw=1.1)
    axis.set(title=title, xlabel='Native snap TRE (mm)', xlim=(0, limit), ylim=(0, 1.02))
    axis.grid(alpha=0.22)
axes[0].set_ylabel('Cumulative distribution')
axes[0].legend(loc='lower right')
fig.suptitle('COPD2 / challenge case 105: P2e error-tail diagnostic', fontsize=14)
plt.show()

pd.DataFrame([
    {'method': method, **summarize(values)}
    for method, values in errors[case].items()
]).set_index('method').round(3)


A left-shifted curve is better. In COPD2, inspect paired landmark indices in the P2e upper tail, their physical locations, and their distance from the crop border before assigning a cause. The CDF establishes a tail failure pattern; it does not, by itself, identify its mechanism.